In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# from rich import print

# def print(*o, sep=' ', end='\n', file=None, flush=False):
#     o = [str(x) for x in o]
#     p(*o, sep=sep, end=end, file=file, flush=flush)

# print.__doc__ = p.__doc__

In [2]:
dataset_path = "dataset/dataset-all-uf100-430.csv"
# dataset_path = "dataset/dataset-uf250-1065.csv"
df = pd.read_csv(dataset_path)

# replace NaN values in 'option' by 'default'
df['option'] = df['option'].fillna('default')
df = df.fillna(0)
# remove the ' ms' from the 'Time' column and convert it to float
df['Propagation'] = df['Propagation'] # - df["Skipped Propagation"] - df["Replayed Propagation"]
df['Learned Clause'] = df['Clause added'] - int(dataset_path.split('-')[-1].split('.')[0]) # remove the initial clauses
# df['Learned Proportion'] = (df['Clause added']) / (df['Clause added'] + df['Forward Subsumption'] + df['Forward Subsumption (learned)'] + df['Backward Subsumption'])
print(df.columns)

Index(['file', 'option', 'Time (ms)', 'Notifications', 'Done', 'Clause added',
       'Decision', 'Implication', 'Unassignement', 'Propagation', 'Conflict',
       'Sync unassign', 'Repairing conflicts', 'Sync assign', 'Time',
       'Clause deleted', 'Forward Subsumption (learned)',
       'Backward Subsumption', 'Restart', 'Skipped Propagation',
       'Replayed Propagation', 'Failed learning', 'Forward Subsumption',
       'Redundant clause', 'Blocker set', 'Learned Clause'],
      dtype='object')


In [3]:
# add the scaled down columns (divided by 1000, 1000000) if they exist
for col in df.columns:
    if col in ['file', 'option']:
        continue
    # if numeric column
    if pd.api.types.is_numeric_dtype(df[col]):
        if df[col].max() > 1000000:
            df[col + ' x10^6'] = df[col] / 1000000
        if df[col].max() > 1000:
            df[col + ' x10^3'] = df[col] / 1000

if "Time (ms) x10^3" in df.columns:
    df = df.rename(columns={"Time (ms) x10^3": "Time (s)"})

In [4]:
# print(f"Average number of cross implication for decision filtered {filtered[filtered['option'] == '-gb']['Cross implication for decision'].mean()}")
# print(f"Average number of cross implication for decision df {df[df['option'] == '-gb']['Cross implication for decision'].mean()}")

In [5]:
options = df['option'].unique()

df_options = {opt: df[df['option'] == opt] for opt in options}


In [6]:
is_unsat = 'uuf' in dataset_path
n_vars = dataset_path.split('uf')[-1].split('-')[0]
if is_unsat:
    print("UNSAT", end=' ')
else:
    print("SAT", end=' ')
print(f"dataset with {int(len(df) / len(df_options))} problems with {n_vars} variables.")
columns = ["option"]
for col in df.columns:
    if col not in ['file', 'option']:
        columns.append(col + " mean")
        columns.append(col + " median")

stats = pd.DataFrame(columns=columns)
for opt, df in df_options.items():
    row = {}
    row['option'] = opt
    for col in df.columns:
        if col not in ['file', 'option']:
            row[col + " mean"] = df[col].mean()
            row[col + " median"] = df[col].median()
    stats.loc[-1] = row
    stats.index = stats.index + 1

# add a boolean column for each option "-gb", "--restarts off"
stats["-gb"] = stats["option"].str.contains("-gb").astype(int)
stats["-lcm"] = stats["option"].str.contains("-lcm").astype(int)
stats["-bl"] = stats["option"].str.contains("-bl").astype(int)
stats["restarts"] = 1 - stats["option"].str.contains("--restarts off").astype(int)
# add a column for whether -ecr, -pcr or neither is used
def get_cr(option: str) -> str:
    if "-ecr" in option:
        return "2"
    elif "-pcr" in option:
        return "1"
    else:
        return "0"
stats["conflict search"] = stats["option"].apply(get_cr)

option_columns = ["-gb", "-lcm", "-bl", "restarts", "conflict search"]

stats = stats.sort_values(by=["restarts", "-bl", "-gb", "-lcm", "conflict search"])
stats = stats.reset_index(drop=True)
stats = stats[["option"] + option_columns + ['Time (ms) mean', 'Sync assign x10^3 mean', 'Propagation x10^3 mean', 'Conflict mean', 'Failed learning mean']]
stats

SAT dataset with 1000 problems with 100 variables.


,option,-gb,-lcm,-bl,restarts,conflict search,Time (ms) mean,Sync assign x10^3 mean,Propagation x10^3 mean,Conflict mean,Failed learning mean
0,--restarts off,0,0,0,0,0,3.792,0.807724,3.548612,150.149,0.000
1,-pcr --restarts off,0,0,0,0,1,5.874,0.790206,4.734620,544.442,0.000
2,-ecr --restarts off,0,0,0,0,2,14.295,0.751651,9.922077,2322.174,0.000
3,-gb --restarts off,1,0,0,0,0,9.356,0.741738,6.929004,173.366,0.014
4,-gb -pcr --restarts off,1,0,0,0,1,12.914,0.708400,8.190842,641.659,0.278
5,-gb -ecr --restarts off,1,0,0,0,2,26.206,0.647659,13.747750,2684.298,0.361
6,-gb -lcm --restarts off,1,1,0,0,0,9.472,0.741738,6.929004,173.366,0.014
7,-gb -lcm -pcr --restarts off,1,1,0,0,1,13.038,0.708400,8.190842,641.659,0.278
8,-gb -lcm -ecr --restarts off,1,1,0,0,2,26.499,0.647659,13.747750,2684.298,0.361
9,-gb -bl --restarts off,1,0,1,0,0,9.329,0.741738,6.929004,173.366,0.014


In [7]:
# compute the correlation with each of the option columns
correlations = {}
for col in option_columns:
    correlations[col] = stats[col].corr(stats['Sync assign x10^3 mean'])
correlations = dict(sorted(correlations.items(), key=lambda item: abs(item[1]), reverse=True))
print("Correlation with Sync assign x10^3 mean:")
for col, corr in correlations.items():
    print(f"{col}: {corr:.3f}")

Correlation with Sync assign x10^3 mean:
restarts: 0.966
-gb: -0.183
conflict search: -0.161
-lcm: -0.075
-bl: -0.075


In [8]:
# split stat in table depending on whether the contain --restarts off, -del off or neither
stat_norestart = stats[stats['option'].str.contains('--restarts off') & ~stats['option'].str.contains('-del off')]
stat_deloff = stats[stats['option'].str.contains('-del off') & ~stats['option'].str.contains('--restarts off')]
stat_both = stats[stats['option'].str.contains('--restarts off') & stats['option'].str.contains('-del off')]
stat_other = stats[~stats['option'].str.contains('--restarts off') & ~stats['option'].str.contains('-del off')]

# in all options, remove the --restarts off and -del off part for better readability
stat_norestart['option'] = stat_norestart['option'].str.replace('--restarts off)', '').str.strip()
stat_deloff['option']    = stat_deloff['option'].str.replace('-del off', '').str.strip()
stat_both['option']      = stat_both['option'].str.replace('--restarts off', '').str.replace('-del off', '').str.strip()
stat_other['option']     = stat_other['option'].str.strip()

stat_split = {
    "Default": stat_other,
    "No restart": stat_norestart,
    # "Deletion off": stat_deloff,
    # "No restart & Deletion off": stat_both,
}

for name, stat in stat_split.items():
    # remove the mean and median suffix for better readability
    stat.columns = [col.replace(' mean', '').replace(' median', '') for col in stat.columns]
    stat.columns = [col.replace('Forward', 'Fw').replace('Backward', 'Bw') for col in stat.columns]

for name, stat in stat_split.items():
    print(f"\n{name}:")
    print(stat.to_markdown(index=False))


Default:
| option            |   -gb |   -lcm |   -bl |   restarts |   conflict search |   Time (ms) |   Sync assign x10^3 |   Propagation x10^3 |   Conflict |   Failed learning |
|:------------------|------:|-------:|------:|-----------:|------------------:|------------:|--------------------:|--------------------:|-----------:|------------------:|
| default           |     0 |      0 |     0 |          1 |                 0 |       5.581 |             1.35389 |             4.93718 |    196.811 |             0     |
| -pcr              |     0 |      0 |     0 |          1 |                 1 |       8.627 |             1.35028 |             6.72067 |    747.309 |             0     |
| -ecr              |     0 |      0 |     0 |          1 |                 2 |      21.18  |             1.30771 |            14.5163  |   3461.33  |             0     |
| -gb               |     1 |      0 |     0 |          1 |                 0 |      12.219 |             1.24305 |             8.35005

/tmp/ipykernel_411208/2811016408.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stat_norestart['option'] = stat_norestart['option'].str.replace('--restarts off)', '').str.strip()
/tmp/ipykernel_411208/2811016408.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stat_other['option']     = stat_other['option'].str.strip()


In [9]:
#create box plots for Sync assign